# Practical 3C — Fashion-MNIST Augmentation

## Aim
Study augmentation on Fashion-MNIST using a deep multilayer perceptron.

## Setup
Run the installation cell if required. These notebooks use Python 3.10–3.12 where possible. On Apple Silicon, TensorFlow/PyTorch acceleration depends on the installed build; CPU execution is supported.

**Important:** dataset downloads and model training may take time. Results are generated when you run the notebook; no accuracy values are pre-filled.

In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn tensorflow

## Reproducibility\nA fixed random seed is used where practical. If a dataset download fails, check your internet connection and rerun the loading cell.

## 1. Load Fashion-MNIST and prepare splits

In [ ]:
import time, numpy as np, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
np.random.seed(42); tf.random.set_seed(42)
(x_all,y_all),(x_test,y_test)=keras.datasets.fashion_mnist.load_data()
x_all=x_all.astype("float32")/255.; x_test=x_test.astype("float32")/255.
x_train,x_val=x_all[5000:],x_all[:5000]; y_train,y_val=y_all[5000:],y_all[:5000]
print(x_train.shape,x_val.shape,x_test.shape)

## 2. Model and gradient utilities

In [ ]:
def make_model(mode="poor"):
    init="random_normal" if mode=="poor" else ("glorot_uniform" if mode=="xavier" else "he_normal")
    model=keras.Sequential([layers.Input((28,28)),layers.Flatten()])
    for _ in range(5):
        model.add(layers.Dense(256,activation="relu",kernel_initializer=init))
    model.add(layers.Dense(10,activation="softmax"))
    model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
    return model

def gradient_norms(model, x, y):
    with tf.GradientTape() as tape:
        pred=model(x,training=True)
        loss=keras.losses.sparse_categorical_crossentropy(y,pred)
        loss=tf.reduce_mean(loss)
    grads=tape.gradient(loss,model.trainable_variables)
    return [float(tf.norm(g)) for g in grads if g is not None]

## 3. Data augmentation and early stopping

In [ ]:
# Images remain 28x28. RandomFlip/Rotation are included in the model pipeline.
augment=keras.Sequential([layers.RandomRotation(0.08),layers.RandomTranslation(0.05,0.05)])
def build_augmented(use_aug):
    inp=keras.Input((28,28)); x=augment(inp) if use_aug else inp
    x=layers.Flatten()(x)
    for _ in range(3): x=layers.Dense(256,activation="relu")(x)
    out=layers.Dense(10,activation="softmax")(x)
    m=keras.Model(inp,out); m.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
    return m
runs={}
for name,use_aug in [("baseline",False),("augmentation",True)]:
    m=build_augmented(use_aug)
    cb=keras.callbacks.EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True)
    h=m.fit(x_train,y_train,validation_data=(x_val,y_val),epochs=50,batch_size=256,callbacks=[cb],verbose=0)
    runs[name]=(m,h); print(name,"epochs:",len(h.history["loss"]),"best val acc:",max(h.history["val_accuracy"]))

In [ ]:
for k,(m,h) in runs.items(): plt.plot(h.history["val_loss"],label=k)
plt.title("Validation loss: baseline vs augmentation"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.show()

**Note:** the source practical asks for random rotations and flips. Horizontal flips can change the semantic identity of some clothing items, so this notebook uses small rotations and translations instead of blindly flipping. If your instructor requires flips, add `layers.RandomFlip('horizontal')` and discuss this limitation.\n\n**Conclusion:** Early stopping halts when validation loss stops improving for five epochs. Compare validation-error trends.